In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from crewai import LLM

llm = LLM(
    model="gemini/gemini-3.1-flash-lite-preview",
    temperature=0.1
)
llm.call("Who invented crew AI framework.")

In [ ]:
from crewai.tools import BaseTool
from crewai import Agent, Task, Crew
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3
)

class JargonSimplifierTool(BaseTool):
    name: str = "Jargon Simplifier Tool"
    description: str = "Replaces internal jargon with business-friendly language."

    def _run(self, text: str) -> str:

        replacements = {
            "PRX": "Project Phoenix AI initiative",
            "TAS": "technical architecture stack",
            "DBX": "enterprise database cluster",
            "SDS": "Smart Data Sync Service",
            "SYNCBOT": "internal workflow assistant",
            "POC": "proof of concept",
            "ETA": "expected completion timeline",
            "ping": "reach out",
            "WIP": "work in progress"
        }

        updated_text = text

        for jargon, replacement in replacements.items():
            updated_text = updated_text.replace(jargon, replacement)

        return updated_text


class ToneAnalyzerTool(BaseTool):
    name: str = "Tone Analyzer Tool"
    description: str = "Checks whether communication tone is professional."

    def _run(self, text: str) -> str:

        aggressive_words = ["urgent", "immediately", "delay"]

        found = []

        for word in aggressive_words:
            if word.lower() in text.lower():
                found.append(word)

        if found:
            return f"Aggressive wording detected: {', '.join(found)}"

        return "Tone is professional."


class ExecutiveSummaryTool(BaseTool):
    name: str = "Executive Summary Tool"
    description: str = "Creates executive-level summaries."

    def _run(self, text: str) -> str:

        return f"""
Executive Summary:
- Incident analyzed
- Key action items identified
- Risks summarized
- Leadership-ready communication generated

Total Words: {len(text.split())}
"""

In [ ]:
technical_email = """
Looping in Priya. TAS and PRX updates are in the deck.
ETA for SDS integration is Friday.
Let's sync tomorrow if SYNCBOT allows.
Ping me if blockers remain.
"""

incident_report = """
DBX outage impacted 3 regions.
Root cause analysis is WIP.
Need POC validation before production rollout ASAP.
"""

email_agent = Agent(
    role="Professional Email Writer",
    goal="Rewrite informal communication professionally",
    backstory="Enterprise communication expert",
    verbose=True,
    tools=[
        JargonSimplifierTool(),
        ToneAnalyzerTool()
    ],
    llm=llm
)

executive_agent = Agent(
    role="Executive Summary Specialist",
    goal="Create concise summaries for leadership",
    backstory="Expert in executive communication",
    verbose=True,
    tools=[
        ExecutiveSummaryTool()
    ],
    llm=llm
)

email_task = Task(
    description=f"""
Rewrite the following email professionally.

EMAIL:
{technical_email}
""",
    agent=email_agent,
    expected_output="Professional email output"
)

incident_task = Task(
    description=f"""
Create an executive summary for leadership.

REPORT:
{incident_report}
""",
    agent=executive_agent,
    expected_output="Executive summary output"
)

In [ ]:
crew = Crew(
    agents=[
        email_agent,
        executive_agent
    ],
    tasks=[
        email_task,
        incident_task
    ],
    verbose=True
)

result = crew.kickoff()

print(result)